In [217]:
import pandas as pd
import plotly.express as px
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix,accuracy_score

# Import data

In [218]:
datasets = pd.read_csv('car_stolen.csv')
datasets.drop(['id'],axis=1,inplace=True)

checked_car_ins = {
    'color': ['Red'],
    'car_type': ['SUV'],
    'origin': ['Domestic']
}
risk_car = pd.DataFrame(checked_car_ins)
risk_car

,color,car_type,origin
0,Red,SUV,Domestic


In [219]:
datasets

,color,car_type,origin,stolen
0,Red,Sport,Domestic,Yes
1,Red,Sport,Domestic,No
2,Red,Sport,Domestic,Yes
3,Yellow,Sport,Domestic,No
4,Yellow,Sport,Imported,Yes
5,Yellow,SUV,Imported,No
6,Yellow,SUV,Imported,Yes
7,Yellow,SUV,Domestic,No
8,Red,SUV,Imported,No
9,Red,Sport,Imported,Yes


In [220]:
datasets.dtypes

color       object
car_type    object
origin      object
stolen      object
dtype: object

In [221]:
datasets

,color,car_type,origin,stolen
0,Red,Sport,Domestic,Yes
1,Red,Sport,Domestic,No
2,Red,Sport,Domestic,Yes
3,Yellow,Sport,Domestic,No
4,Yellow,Sport,Imported,Yes
5,Yellow,SUV,Imported,No
6,Yellow,SUV,Imported,Yes
7,Yellow,SUV,Domestic,No
8,Red,SUV,Imported,No
9,Red,Sport,Imported,Yes


# Pre-Convert for more easy.

In [222]:
# Covert from here
from sklearn.preprocessing import LabelEncoder
color_mapping = {'Yellow': 0, 'Red': 1}
car_type_mapping = {'Sport': 0, 'SUV': 1}
origin_mapping = {'Imported': 0, 'Domestic': 1}
stolen_mapping = {'No': 0, 'Yes': 1}


risk_car['numeric-color'] = risk_car['color'].map(color_mapping)
risk_car['numeric-car_type'] = risk_car['car_type'].map(car_type_mapping)
risk_car['numeric-origin'] = risk_car['origin'].map(origin_mapping)


In [223]:
label_encoder = LabelEncoder()

datasets['numeric-color'] = datasets['color'].map(color_mapping)
datasets['numeric-car_type'] = datasets['car_type'].map(car_type_mapping)
datasets['numeric-origin'] = datasets['origin'].map(origin_mapping)
datasets['numeric-stolen'] = datasets['stolen'].map(stolen_mapping)

datasets

,color,car_type,origin,stolen,numeric-color,numeric-car_type,numeric-origin,numeric-stolen
0,Red,Sport,Domestic,Yes,1,0,1,1
1,Red,Sport,Domestic,No,1,0,1,0
2,Red,Sport,Domestic,Yes,1,0,1,1
3,Yellow,Sport,Domestic,No,0,0,1,0
4,Yellow,Sport,Imported,Yes,0,0,0,1
5,Yellow,SUV,Imported,No,0,1,0,0
6,Yellow,SUV,Imported,Yes,0,1,0,1
7,Yellow,SUV,Domestic,No,0,1,1,0
8,Red,SUV,Imported,No,1,1,0,0
9,Red,Sport,Imported,Yes,1,0,0,1


In [224]:
datasets.dtypes

color               object
car_type            object
origin              object
stolen              object
numeric-color        int64
numeric-car_type     int64
numeric-origin       int64
numeric-stolen       int64
dtype: object

In [225]:
risk_car

,color,car_type,origin,numeric-color,numeric-car_type,numeric-origin
0,Red,SUV,Domestic,1,1,1


In [226]:
risk_car.dtypes

color               object
car_type            object
origin              object
numeric-color        int64
numeric-car_type     int64
numeric-origin       int64
dtype: object

In [227]:
datasets.drop(['color','car_type','origin','stolen'],axis=1,inplace=True)
risk_car.drop(['color','car_type','origin'],axis=1,inplace=True)

In [228]:
datasets

,numeric-color,numeric-car_type,numeric-origin,numeric-stolen
0,1,0,1,1
1,1,0,1,0
2,1,0,1,1
3,0,0,1,0
4,0,0,0,1
5,0,1,0,0
6,0,1,0,1
7,0,1,1,0
8,1,1,0,0
9,1,0,0,1


In [229]:
risk_car

,numeric-color,numeric-car_type,numeric-origin
0,1,1,1


In [230]:
y_df = datasets[['numeric-stolen']]
datasets.drop(['numeric-stolen'],axis=1,inplace=True)
X_df = datasets
with pd.option_context('display.max_rows', 8): display(X_df)

,numeric-color,numeric-car_type,numeric-origin
0,1,0,1
1,1,0,1
2,1,0,1
3,0,0,1
...,...,...,...
6,0,1,0
7,0,1,1
8,1,1,0
9,1,0,0


In [231]:
X = X_df.values
y = y_df.values

# Preprocess

In [232]:
scaler = StandardScaler()
X_norm = scaler.fit_transform(X)
new = risk_car.values
new = scaler.transform(new)

# Find Best k K-NN

In [233]:
X_train , X_test , y_train , y_test = train_test_split(X,y,train_size=0.6,random_state=1992)

In [234]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [235]:
scores = []
for k in range(1,7):
  neigh = KNeighborsClassifier(n_neighbors=k,metric='euclidean')
  neigh.fit(X_train, y_train.ravel())
  scores.append([k,neigh.score(X_test,y_test)])
print(scores)

[[1, 0.25], [2, 0.0], [3, 0.0], [4, 0.0], [5, 0.0], [6, 0.0]]


In [236]:
scores.sort(key=lambda x: x[1], reverse=True)
print(f"Best k={scores[0][0]} with accuracy score:{scores[0][1]}")

Best k=1 with accuracy score:0.25


# K-NN k=1

In [237]:
neigh = KNeighborsClassifier(n_neighbors=1,metric='euclidean')
neigh.fit(X_norm, y)

/usr/local/lib/python3.10/dist-packages/sklearn/neighbors/_classification.py:215: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return self._fit(X, y)


KNeighborsClassifier(metric='euclidean', n_neighbors=1)

In [238]:
y_pred = neigh.predict(new)
print(y_pred)

[0]


# Using Best k & Show Confusion Matrix

In [239]:
k = scores[0][0]
neigh = KNeighborsClassifier(n_neighbors=k,metric='euclidean')
neigh.fit(X_train, y_train.ravel())

KNeighborsClassifier(metric='euclidean', n_neighbors=1)

In [240]:
y_pred = neigh.predict(X_test)
cm = confusion_matrix(y_test, y_pred)
print(cm)
print(classification_report(y_test,y_pred))

[[0 0]
 [3 1]]
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         0
           1       1.00      0.25      0.40         4

    accuracy                           0.25         4
   macro avg       0.50      0.12      0.20         4
weighted avg       1.00      0.25      0.40         4



/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [241]:
y_pred = neigh.predict(new)
print(y_pred)

[0]
